# Chapter 04 — Scrum Built the Training Set

**Companion to *Applied AI*.**

This notebook accompanies Chapter 4. The chapter argues that software automated
itself first because of **checkability**, not difficulty. This notebook makes
the accounting behind that claim runnable.

## Question

In a generate-and-filter pipeline, **how much of the usable correctness comes
from the generator, and how much from the verifier?**

## What this notebook establishes

- The SWE-bench harvest yield recomputed from the chapter's own figures: what a
  strict four-filter verifier costs when applied honestly.
- A weak generator behind a cheap exact verifier: the filter removes almost
  everything and contributes essentially all of the correctness.
- Why an attempt **without** a verdict teaches nothing: a discriminator trained
  on successes only cannot tell good from bad.
- The six-question terrain survey, run as code over the chapter's four
  candidate domains, including the one that fails.

## What this notebook does **not** establish

- The generator here is a local random process, **not a language model**. It
  reproduces the *shape* of the accounting, not anyone's pass rate.
- The SWE-bench and AlphaCode figures are **quoted from the chapter**, which
  cites them from the original papers. Nothing here re-runs those benchmarks.
- The terrain survey scores are the chapter's editorial judgments, entered as
  data. They are arguments, not measurements.

## Setup

In [1]:
import random
from dataclasses import dataclass

SEED = 42
random.seed(SEED)

## 1. The receipt: what a strict verifier costs

The chapter reports SWE-bench being built from roughly 90,000 pull requests,
filtered on four criteria, leaving 2,294 task instances.

In [2]:
PULL_REQUESTS = 90_000        # quoted from the chapter
INSTANCES     = 2_294         # quoted from the chapter

yield_rate = INSTANCES / PULL_REQUESTS
print(f"pull requests examined : {PULL_REQUESTS:,}")
print(f"task instances kept    : {INSTANCES:,}")
print(f"harvest yield          : {yield_rate:.4f}  ({yield_rate:.1%})")
print(f"discarded by filters   : {1 - yield_rate:.1%}")

FILTERS = [
    ("resolves a GitHub issue", "selects for a written specification"),
    ("contributes tests",       "selects for a written verifier"),
    ("the project installs",    "selects for a runnable environment"),
    ("fail-to-pass transition", "confirms the verifier discriminates"),
]
print()
for name, job in FILTERS:
    print(f"  {name:<26} -> {job}")

pull requests examined : 90,000
task instances kept    : 2,294
harvest yield          : 0.0255  (2.5%)
discarded by filters   : 97.5%

  resolves a GitHub issue    -> selects for a written specification
  contributes tests          -> selects for a written verifier
  the project installs       -> selects for a runnable environment
  fail-to-pass transition    -> confirms the verifier discriminates


That ~2.5% is not a target. It is what a strict verifier costs when applied
honestly — and the chapter's point is that nobody paid that cost *for the
benchmark*. Developers paid it to close tickets. The benchmark is a harvest.

## 2. A weak generator behind a cheap exact verifier

Now the mechanism. The task: repair a function so it returns the mean of the
list it is given. The verifier is the hidden test, and it is exact and free.

In [3]:
def reference(xs):
    return sum(xs) / len(xs)

CANDIDATES = {
    "correct":        lambda xs: sum(xs) / len(xs),
    "off_by_one":     lambda xs: sum(xs) / (len(xs) - 1),
    "sum_only":       lambda xs: sum(xs),
    "first_element":  lambda xs: xs[0],
    "int_divide":     lambda xs: sum(xs) // len(xs),
    "crashes":        lambda xs: 1 / 0,
}

# A weak generator: mostly wrong, occasionally right.
WEIGHTS = {
    "correct": 0.06, "off_by_one": 0.24, "sum_only": 0.24,
    "first_element": 0.22, "int_divide": 0.18, "crashes": 0.06,
}

def generate(rng):
    name = rng.choices(list(WEIGHTS), weights=list(WEIGHTS.values()))[0]
    return name, CANDIDATES[name]

TESTS = [[1, 2, 3], [4, 4, 4, 4], [10, 20], [7], [1, 2]]
# NOTE: [1, 2] is load-bearing. Without it the suite accepts integer
# division, because every other case happens to divide evenly. See the
# adequacy note below - this was a real defect in this notebook.

def verify(fn) -> bool:
    """Deterministic, exact, free. The chapter's cheap check."""
    try:
        return all(abs(fn(xs) - reference(xs)) < 1e-9 for xs in TESTS)
    except Exception:
        return False

In [4]:
rng = random.Random(SEED)
N = 20_000

proposals = [generate(rng) for _ in range(N)]
accepted  = [(n, f) for n, f in proposals if verify(f)]

print(f"candidates generated : {N:,}")
print(f"accepted by verifier : {len(accepted):,}  ({len(accepted)/N:.2%})")
print(f"removed by verifier  : {N - len(accepted):,}  ({1 - len(accepted)/N:.2%})")
print()
print("every accepted candidate is correct on every hidden test:")
assert all(verify(f) for _, f in accepted)
print("  assertion held")
print()
from collections import Counter
print("what the generator actually produced:")
for name, k in Counter(n for n, _ in proposals).most_common():
    kept = sum(1 for n, _ in accepted if n == name)
    print(f"  {name:<15} proposed {k:>6,}   accepted {kept:>5,}")

candidates generated : 20,000
accepted by verifier : 1,142  (5.71%)
removed by verifier  : 18,858  (94.29%)

every accepted candidate is correct on every hidden test:
  assertion held

what the generator actually produced:
  off_by_one      proposed  4,802   accepted     0
  sum_only        proposed  4,766   accepted     0
  first_element   proposed  4,449   accepted     0
  int_divide      proposed  3,616   accepted     0
  crashes         proposed  1,225   accepted     0
  correct         proposed  1,142   accepted 1,142


## Observation

Read that as an accounting statement, the way the chapter reads AlphaCode's.

In [5]:
gen_rate = len(accepted) / N
print(f"generator's raw acceptable rate      : {gen_rate:.2%}")
print(f"accepted set, w.r.t. these tests     : 100.00% passing")
print("   (correct *as far as the tests reach* - not correctness in general)")
print()
print("The generator supplied VARIANCE.")
print("The verifier supplied CORRECTNESS, at zero cost and with no intelligence.")
print()
print(f"AlphaCode, quoted from the chapter: example-test filtering removed")
print(f"MORE THAN 99% of samples. Here the filter removed {1 - gen_rate:.2%}.")

generator's raw acceptable rate      : 5.71%
accepted set, w.r.t. these tests     : 100.00% passing
   (correct *as far as the tests reach* - not correctness in general)

The generator supplied VARIANCE.
The verifier supplied CORRECTNESS, at zero cost and with no intelligence.

AlphaCode, quoted from the chapter: example-test filtering removed
MORE THAN 99% of samples. Here the filter removed 94.29%.


The lesson the chapter draws is the practical one:

> If you are deciding where to invest in a new domain, and the domain's
> verifier is weak, then buying a better model is buying more of the component
> that was not producing the correctness.

Watch that directly: improve the generator, then weaken the verifier.

In [6]:
def pipeline(gen_quality, verifier_fn, n=20_000, seed=SEED):
    r = random.Random(seed)
    w = dict(WEIGHTS)
    w["correct"] = gen_quality
    rest = 1 - gen_quality
    scale = rest / (sum(v for k, v in WEIGHTS.items() if k != "correct"))
    for k in w:
        if k != "correct":
            w[k] = WEIGHTS[k] * scale
    names = list(w)
    out = []
    for _ in range(n):
        nm = r.choices(names, weights=[w[k] for k in names])[0]
        out.append((nm, CANDIDATES[nm]))
    acc = [(nm, f) for nm, f in out if verifier_fn(f)]
    truly_ok = sum(1 for nm, _ in acc if nm == "correct")
    return len(acc), truly_ok

def weak_verifier(fn):
    """Only checks it runs and returns a number. A plausible bad check."""
    try:
        v = fn([1, 2, 3])
        return isinstance(v, (int, float))
    except Exception:
        return False

print(f"{'generator quality':<20}{'strong verifier':<34}{'weak verifier'}")
print("-" * 86)
for q in (0.06, 0.20, 0.50, 0.90):
    a_s, ok_s = pipeline(q, verify)
    a_w, ok_w = pipeline(q, weak_verifier)
    s = f"{a_s:,} accepted / {ok_s:,} correct"
    wv = f"{a_w:,} accepted / {ok_w:,} correct ({ok_w/max(a_w,1):.0%} right)"
    print(f"{q:<20.0%}{s:<34}{wv}")

generator quality   strong verifier                   weak verifier
--------------------------------------------------------------------------------------
6%                  1,142 accepted / 1,142 correct    18,775 accepted / 1,142 correct (6% right)
20%                 4,067 accepted / 4,067 correct    18,974 accepted / 4,067 correct (21% right)
50%                 9,931 accepted / 9,931 correct    19,355 accepted / 9,931 correct (51% right)


90%                 17,997 accepted / 17,997 correct  19,878 accepted / 17,997 correct (91% right)


The strong-verifier column is always 100% correct; only its *throughput*
changes with generator quality. The weak-verifier column accepts far more and
is mostly wrong, and **a better generator does not repair it** — it just
supplies more well-formed wrong answers to a check that cannot see the
difference.

### An adequacy defect, found while writing this notebook

The first draft of this notebook used four tests: `[1,2,3]`, `[4,4,4,4]`,
`[10,20]` and `[7]`. Every one of them divides evenly, so a candidate computing
`sum(xs) // len(xs)` — **integer** division — passed all four and was accepted
3,616 times out of 20,000.

The verifier was cheap, exact, deterministic and independent of the generator.
It was also **inadequate**: it never tested the property that separates integer
division from real division. Adding `[1, 2]` fixes it.

That is Chapter 21's distinction arriving seventeen chapters early, inside a
notebook written to celebrate verifiers:

```text
a check that runs   !=   a check adequate to the property you care about
```

In [7]:
NARROW = [[1, 2, 3], [4, 4, 4, 4], [10, 20], [7]]     # the original, inadequate
WIDE   = TESTS                                         # with [1, 2] added

def verify_with(tests, fn):
    try:
        return all(abs(fn(xs) - reference(xs)) < 1e-9 for xs in tests)
    except Exception:
        return False

print(f"{'candidate':<16}{'narrow suite':<16}{'wide suite'}")
print("-" * 48)
for name, fn in CANDIDATES.items():
    a = "accepted" if verify_with(NARROW, fn) else "rejected"
    b = "accepted" if verify_with(WIDE, fn) else "rejected"
    flag = "   <-- the defect" if a != b else ""
    print(f"{name:<16}{a:<16}{b}{flag}")

print()
print("int_divide is wrong. Four honest tests could not see it.")

candidate       narrow suite    wide suite
------------------------------------------------
correct         accepted        accepted
off_by_one      rejected        rejected
sum_only        rejected        rejected
first_element   rejected        rejected
int_divide      accepted        rejected   <-- the defect
crashes         rejected        rejected

int_divide is wrong. Four honest tests could not see it.


## 3. Failures are data too

SWE-Gym trained its verifier on **1,318 passing and 1,318 failing** trajectories
(quoted from the chapter). Half the signal was failure. Here is why that is not
an accident.

In [8]:
def train_threshold(labelled):
    """A discriminator is a boundary. It needs examples on both sides."""
    passing = [score for score, ok in labelled if ok]
    failing = [score for score, ok in labelled if not ok]
    if not passing or not failing:
        return None, "cannot place a boundary: only one class present"
    return (max(failing) + min(passing)) / 2, "boundary placed between the classes"

rng2 = random.Random(SEED)
labelled_both = [(rng2.uniform(0.6, 1.0), True) for _ in range(40)] + \
                [(rng2.uniform(0.0, 0.5), False) for _ in range(40)]
labelled_wins = [(s, ok) for s, ok in labelled_both if ok]

for name, data in (("successes AND failures", labelled_both),
                   ("successes only", labelled_wins)):
    t, note = train_threshold(data)
    shown = f"{t:.3f}" if t is not None else "NONE"
    print(f"{name:<24} threshold={shown:<8} {note}")

successes AND failures   threshold=0.549    boundary placed between the classes
successes only           threshold=NONE     cannot place a boundary: only one class present


> **An attempt with a verdict is an example, whichever way the verdict went.
> An attempt without a verdict is only text.**

A record that keeps what shipped and throws away what did not has kept the half
that cannot teach the difference.

## 4. The terrain survey, as code

The chapter's six questions, applied to its four candidate domains. The scores
below are **the chapter's editorial judgments entered as data**, not
measurements.

In [9]:
QUESTIONS = [
    "already text or symbols",
    "decomposes into attemptable units",
    "verifier cheaper than the work",     # load-bearing
    "wrong proposal cheap to discard",
    "attempts and verdicts recorded",
    "volume with variation",
]
LOAD_BEARING = 2

DOMAINS = {
    "software engineering":   [True, True, True,  True,  True,  True],
    "research synthesis":     [True, True, True,  True,  False, True],
    "adjudication / claims":  [True, True, True,  False, True,  True],
    "evaluation / grading":   [True, True, True,  True,  False, True],
    "fraud detection":        [True, True, False, True,  True,  True],
}

NOTES = {
    "research synthesis":    "verifier is real: a cited source contains the claim or it does not",
    "adjudication / claims": "authority-heavy: a wrong proposal already sent is not cheap to discard",
    "evaluation / grading":  "trap: the 'verifier' is often just another judgment (Ch 21)",
    "fraud detection":       "verifier is DELAYED and ADVERSARIAL - fails the load-bearing row",
}

print(f"{'domain':<26}{'score':<9}{'verifier row':<16}verdict")
print("-" * 92)
for name, answers in DOMAINS.items():
    score = sum(answers)
    ok = answers[LOAD_BEARING]
    verdict = "process possible" if ok else "DEMO, NOT A PROCESS"
    print(f"{name:<26}{score}/6      {'yes' if ok else 'NO':<16}{verdict}")
print()
for name, note in NOTES.items():
    print(f"  {name:<24} {note}")

domain                    score    verifier row    verdict
--------------------------------------------------------------------------------------------
software engineering      6/6      yes             process possible
research synthesis        5/6      yes             process possible
adjudication / claims     5/6      yes             process possible
evaluation / grading      5/6      yes             process possible
fraud detection           5/6      NO              DEMO, NOT A PROCESS

  research synthesis       verifier is real: a cited source contains the claim or it does not
  adjudication / claims    authority-heavy: a wrong proposal already sent is not cheap to discard
  evaluation / grading     trap: the 'verifier' is often just another judgment (Ch 21)
  fraud detection          verifier is DELAYED and ADVERSARIAL - fails the load-bearing row


## Interpretation

Fraud detection is the row that matters. It scores 5 of 6 — text, decomposable,
cheap to discard, recorded, enormous volume — and fails on the one question
that decides deployability. On a naive reading it looks like the best candidate
on the list.

> **A survey that returns "yes" for everything is not a survey.**

The chapter's conclusion follows from the two halves of this notebook. The
verifier does most of the work, so the first job in a new domain is usually not
to call a model:

```text
text + deduction            -> a demo
text + deduction + a check  -> a process
a process that keeps its verdicts -> one that can improve
```

## Try it yourself

1. **Set the verifier free but delayed.** Make `verify` return `None` for 90
   days before returning a verdict. What can the pipeline do in the meantime,
   and what is your accepted set worth?
2. **Score your own domain** by adding a row to `DOMAINS`. If you cannot name
   the verifier, write `False` — the chapter's instruction is to write "none"
   rather than to invent one.
3. **Find the next adequacy hole.** The wide suite still accepts anything that
   agrees on five lists. Write a candidate that passes all five and is still
   wrong. It is easier than it sounds, and it is why Chapter 21 treats
   adequacy as unenforceable at the boundary.
4. **Recompute the yield** if a fifth filter were added. At what yield does
   harvesting stop being cheaper than authoring the tasks?